# HDDM 参数拟合工作流

**项目**: GP-SPE 实验设计优化 — Self-Matching Task DDM 参数提取

**工作流总览**:
1. **Step 1** (本机) → 数据预处理：原始 CSV → HDDM 就绪 CSV
2. **Step 2** (Docker) → HDDM 层级模型拟合
3. **Step 3** (本机) → 迹线提取 + 参数可视化

**前置条件**:
- 本机已安装 `pandas`, `numpy`, `matplotlib`
- Docker 已安装并拉取 `hcp4715/hddm` 镜像

## 环境设置 & 路径自动检测

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re
import warnings
warnings.filterwarnings("ignore")

# 自动向上查找 AGENTS.md 定位项目根目录
BASE_DIR = Path.cwd()
while not (BASE_DIR / "AGENTS.md").exists() and BASE_DIR.parent != BASE_DIR:
    BASE_DIR = BASE_DIR.parent
print(f"项目根目录: {BASE_DIR}")
print(f"AGENTS.md 存在: {(BASE_DIR / 'AGENTS.md').exists()}")

RAW_DIR   = BASE_DIR / "2_Data" / "Real_Data" / "UnExtact" / "raw"
READY_DIR = BASE_DIR / "2_Data" / "Real_Data" / "HDDM_Ready"
TRACE_DIR = BASE_DIR / "2_Data" / "Real_Data" / "HDDM_Traces"
FIG_DIR   = BASE_DIR / "3_Figures" / "HDDM_Results"

for d in [READY_DIR, TRACE_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"原始数据:   {RAW_DIR}  存在: {RAW_DIR.exists()}")
print(f"HDDM就绪:   {READY_DIR}")
print(f"HDDM迹线:   {TRACE_DIR}")
print(f"图表输出:   {FIG_DIR}")

项目根目录: /
AGENTS.md 存在: False


PermissionError: [Errno 13] Permission denied: '/2_Data'

---
## Step 1: 数据预处理

从 88 个原始 CSV 中提取 Matching 试次、标记遗漏试次、输出 HDDM 就绪格式。

In [2]:
print("=" * 60)
print("Step 1: 数据预处理 - 准备 HDDM 输入数据")
print("=" * 60)

csv_files = sorted(RAW_DIR.glob("EXP_data_group*.csv"))
print(f"\n发现 {len(csv_files)} 个原始数据文件")

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"合并总行数: {len(df_all)}")

print("\n各组被试分布:")
group_subjects = df_all.groupby("groupID")["subjectID"].nunique()
for gid in sorted(group_subjects.index):
    print(f"  Group {gid}: {group_subjects[gid]} 名被试")

df_f = df_all[df_all["stage"] == "formal"].copy()
print(f"\n过滤 stage='formal' 后: {len(df_f)} 行")

matching_mask = (
    ((df_f["Shape"] == "circle") & (df_f["Label"] == "self"))
    | ((df_f["Shape"] == "square") & (df_f["Label"] == "stranger"))
)
df_match = df_f[matching_mask].copy()
print(f"过滤 Matching 试次后: {len(df_match)} 行")

df_match["identity"] = df_match["Label"].map({"self": 1, "stranger": 0})
df_match["RT_num"] = pd.to_numeric(df_match["RT"], errors="coerce")
df_match["omission"] = df_match["RT_num"].isna().astype(int)
df_match["T_s"] = pd.to_numeric(df_match["T"], errors="coerce")
df_match["W_s"] = pd.to_numeric(df_match["W"], errors="coerce")
df_match["deadline"] = df_match["T_s"] + df_match["W_s"]

df_match["rt"] = np.where(
    df_match["omission"] == 1,
    df_match["deadline"],
    df_match["RT_num"],
)
df_match["response"] = np.where(df_match["Correct"] == 1, 1, 0)

for gid in sorted(df_match["groupID"].unique()):
    gdf = df_match[df_match["groupID"] == gid].copy()
    subj_map = {s: i for i, s in enumerate(sorted(gdf["subjectID"].unique()))}
    gdf["subj_idx"] = gdf["subjectID"].map(subj_map)

    hddm_cols = ["subj_idx", "rt", "response", "identity", "omission"]
    hddm_df = gdf[hddm_cols].copy()

    P_val = int(gdf["P"].iloc[0])
    T_val = int(gdf["T_s"].iloc[0] * 1000)
    W_val = int(gdf["W_s"].iloc[0] * 1000)

    fn = f"hddm_data_group{gid}_P{P_val}_T{T_val}_W{W_val}.csv"
    out_path = READY_DIR / fn
    hddm_df.to_csv(out_path, index=False)

    n_subj = gdf["subjectID"].nunique()
    n_trials = len(hddm_df)
    n_omissions = hddm_df["omission"].sum()
    omission_rate = n_omissions / n_trials * 100
    n_valid = (hddm_df["omission"] == 0).sum()
    acc = hddm_df.loc[hddm_df["omission"] == 0, "response"].mean()

    print(f"\n  Group {gid} (P={P_val}, T={T_val}ms, W={W_val}ms) → {fn}")
    print(f"    被试: {n_subj} | 试次: {n_trials}")
    print(f"    有效: {n_valid} | 遗漏: {n_omissions} ({omission_rate:.1f}%)")
    print(f"    正确率(有效试次): {acc:.3f}")

print(f"\n✅ HDDM 就绪数据已保存到: {READY_DIR}")

Step 1: 数据预处理 - 准备 HDDM 输入数据

发现 88 个原始数据文件
合并总行数: 52320

各组被试分布:
  Group 1: 11 名被试
  Group 2: 12 名被试
  Group 3: 10 名被试
  Group 4: 11 名被试
  Group 5: 11 名被试
  Group 6: 10 名被试
  Group 7: 12 名被试
  Group 8: 11 名被试

过滤 stage='formal' 后: 45760 行
过滤 Matching 试次后: 22880 行

  Group 1 (P=0, T=30ms, W=300ms) → hddm_data_group1_P0_T30_W300.csv
    被试: 11 | 试次: 2860
    有效: 792 | 遗漏: 2068 (72.3%)
    正确率(有效试次): 0.537

  Group 2 (P=0, T=30ms, W=600ms) → hddm_data_group2_P0_T30_W600.csv
    被试: 12 | 试次: 3120
    有效: 1484 | 遗漏: 1636 (52.4%)
    正确率(有效试次): 0.551

  Group 3 (P=120, T=30ms, W=600ms) → hddm_data_group3_P120_T30_W600.csv
    被试: 10 | 试次: 2600
    有效: 1596 | 遗漏: 1004 (38.6%)
    正确率(有效试次): 0.530

  Group 4 (P=120, T=80ms, W=600ms) → hddm_data_group4_P120_T80_W600.csv
    被试: 11 | 试次: 2860
    有效: 1762 | 遗漏: 1098 (38.4%)
    正确率(有效试次): 0.733

  Group 5 (P=8, T=100ms, W=1100ms) → hddm_data_group5_P8_T100_W1100.csv
    被试: 11 | 试次: 2860
    有效: 2547 | 遗漏: 313 (10.9%)
    正确率(有效试次): 0.767

  Gr

---
## Step 2: Docker HDDM 层级模型拟合

### ⚠️ 此步骤需要在 Docker 容器中运行

**方法 A — 使用 Docker_User.html 工具（推荐）**:
1. 用浏览器打开 `Docker_User.html`
2. 输入 `HDDM_Ready` 的完整路径
3. 点击生成命令, 复制到 CMD 运行
4. 在 Jupyter 中新建 Notebook, 逐组运行拟合代码

**方法 B — 手动命令行**:
```powershell
docker pull hcp4715/hddm

docker run -it --rm --cpus=4 `
  -v /d/GitHub_programe/GitHub/Guassion-Process-Experiment-Design:/data `
  -p 8888:8888 `
  hcp4715/hddm `
  jupyter notebook
```

**Docker 内拟合代码** 见下方单元格。将该代码复制到 Docker Jupyter 中运行。

In [1]:
# ⚠️ 此单元格在 Docker 容器内运行
# 复制到 Docker Jupyter Notebook 中执行

import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

BASE_DIR = Path("/data")
DATA_DIR = BASE_DIR / "2_Data" / "Real_Data" / "HDDM_Ready"
OUT_DIR  = BASE_DIR / "2_Data" / "Real_Data" / "HDDM_Traces"
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(DATA_DIR.glob("hddm_data_group*.csv"))
print(f"发现 {len(csv_files)} 个待拟合文件")

for csv_path in csv_files:
    fname = csv_path.stem
    print(f"\n{'=' * 50}")
    print(f"拟合: {fname}")
    print(f"{'=' * 50}")

    df = pd.read_csv(csv_path)
    n_subj = df["subj_idx"].nunique()
    print(f"  被试: {n_subj}, 试次: {len(df)}")
    print(f"  Self: {(df['identity']==1).sum()}, Stranger: {(df['identity']==0).sum()}")
    print(f"  遗漏: {df['omission'].sum()}")

    df_fit = df.copy().dropna(subset=["rt"])
    if len(df_fit) < 50:
        print(f"  ⚠️ 有效试次过少 ({len(df_fit)}), 跳过")
        continue

    import hddm

    model = hddm.HDDM(
        df_fit,
        depends_on={"v": "identity"},
        include=["v", "a", "t", "z"],
        bias=False,
        p_outlier=0.05,
    )

    print("  开始 MCMC 采样...")
    model.sample(3000, burn=500, dbname="traces.db", db="pickle")
    print("  采样完成")

    traces = model.get_traces()
    npz_path = OUT_DIR / f"{fname}_traces.npz"
    trace_dict = {}
    for key, val in traces.items():
        if isinstance(val, np.ndarray):
            trace_dict[key] = val
        elif isinstance(val, pd.DataFrame):
            trace_dict[key] = val.values
    np.savez_compressed(npz_path, **trace_dict)

    stats = model.gen_stats()
    stats.to_csv(OUT_DIR / f"{fname}_stats.csv")

    print(f"  迹线 → {npz_path}")
    print(f"  统计 → {OUT_DIR / f'{fname}_stats.csv'}")

print(f"\n{'=' * 50}")
print("✅ 所有拟合完成!")

PermissionError: [Errno 13] Permission denied: '/data'

---
## Step 3: 参数提取与可视化

**前置条件**: Docker 拟合完成，`HDDM_Traces/` 下存在 `*_traces.npz` 文件。

此步骤**在本机运行**，无需 Docker。

In [ ]:
plt.rcParams["font.sans-serif"] = ["SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("=" * 60)
print("Step 3: 提取 HDDM 参数后验分布")
print("=" * 60)

stats_files = sorted(TRACE_DIR.glob("*_stats.csv"))
npz_files   = sorted(TRACE_DIR.glob("*_traces.npz"))

if len(stats_files) == 0:
    raise FileNotFoundError(
        f"未找到迹线文件，请先在 Docker 中完成 Step 2 拟合。\n"
        f"预期位置: {TRACE_DIR}"
    )

print(f"发现 {len(stats_files)} 个 stats, {len(npz_files)} 个 trace")

all_params = []

for stats_path, npz_path in zip(stats_files, npz_files):
    fname = stats_path.stem.replace("_stats", "")
    print(f"\n处理: {fname}")

    match = re.search(r"group(\d+)_P(\d+)_T(\d+)_W(\d+)", fname)
    if not match:
        continue
    group_id = int(match.group(1))
    P_val = int(match.group(2))
    T_val = int(match.group(3))
    W_val = int(match.group(4))

    stats = pd.read_csv(stats_path, index_col=0)
    traces = dict(np.load(npz_path, allow_pickle=True))
    trace_keys = list(traces.keys())

    for k in trace_keys:
        arr = traces[k]
        if arr.ndim == 0:
            arr = arr.item()
            traces[k] = arr.flatten() if isinstance(arr, np.ndarray) else np.atleast_1d(arr)
        elif arr.ndim > 1:
            traces[k] = arr.flatten()

    row = {"group_id": group_id, "P": P_val, "T_ms": T_val, "W_ms": W_val, "M_ms": T_val + W_val}

    for param_name in ["v", "a", "t", "z"]:
        for key in trace_keys:
            key_lower = key.lower()
            if param_name == key_lower and "subj" not in key_lower:
                s = traces[key]
                row[f"{param_name}_mean"] = float(np.mean(s))
                row[f"{param_name}_std"]  = float(np.std(s))
                row[f"{param_name}_q025"] = float(np.percentile(s, 2.5))
                row[f"{param_name}_q975"] = float(np.percentile(s, 97.5))
                break

    if any(p in trace_keys for p in ["v(identity_1)", "v(identity_0)"]):
        for ikey in ["v(identity_1)", "v(identity_0)"]:
            if ikey in traces:
                label = "v_self" if "1" in ikey else "v_stranger"
                s = traces[ikey]
                row[f"{label}_mean"] = float(np.mean(s))
                row[f"{label}_q025"] = float(np.percentile(s, 2.5))
                row[f"{label}_q975"] = float(np.percentile(s, 97.5))

    if "v_self_mean" in row and "v_stranger_mean" in row:
        row["SPE_v"] = row["v_self_mean"] - row["v_stranger_mean"]

    all_params.append(row)

df_params = pd.DataFrame(all_params).sort_values("group_id")
df_params.to_csv(TRACE_DIR / "all_groups_ddm_params.csv", index=False)

print("\n" + "=" * 60)
print("DDM 参数汇总 (后验均值)")
print("=" * 60)
display_cols = ["group_id","P","T_ms","W_ms","v_self_mean","v_stranger_mean","SPE_v","a_mean","t_mean"]
available = [c for c in display_cols if c in df_params.columns]
display(df_params[available].round(4))

In [ ]:
# -------- 可视化: 各组 DDM 参数 --------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
if "SPE_v" in df_params.columns:
    valid = df_params.dropna(subset=["SPE_v"])
    colors = plt.cm.viridis(np.linspace(0, 1, len(valid)))
    for i, (_, r) in enumerate(valid.iterrows()):
        yerr_lo = r["v_self_mean"] - r["v_self_q025"]
        yerr_hi = r["v_self_q975"] - r["v_self_mean"]
        ax.errorbar(i, r["SPE_v"], yerr=[[yerr_lo], [yerr_hi]],
                    fmt="o", color=colors[i], capsize=5, markersize=8)
    ax.axhline(y=0, color="gray", linestyle="--")
    ax.set_xticks(range(len(valid)))
    ax.set_xticklabels([f"G{r['group_id']:.0f}" for _, r in valid.iterrows()], rotation=45)
    ax.set_ylabel("SPE_v (v_self - v_stranger)")
    ax.set_title("各组 SPE 漂移率效应")

ax = axes[0, 1]
if "v_self_mean" in df_params.columns:
    valid = df_params.dropna(subset=["v_self_mean"])
    x = np.arange(len(valid)); w = 0.35
    ax.bar(x - w/2, valid["v_self_mean"], w, label="Self", alpha=0.8)
    ax.bar(x + w/2, valid["v_stranger_mean"], w, label="Stranger", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([f"G{r['group_id']:.0f}" for _, r in valid.iterrows()], rotation=45)
    ax.set_ylabel("Drift Rate v")
    ax.set_title("Self vs Stranger 漂移率")
    ax.legend()

ax = axes[1, 0]
if "a_mean" in df_params.columns:
    valid = df_params.dropna(subset=["a_mean"])
    x = np.arange(len(valid))
    ax.bar(x, valid["a_mean"], color="steelblue", alpha=0.8)
    ax.errorbar(x, valid["a_mean"],
                yerr=[valid["a_mean"]-valid["a_q025"], valid["a_q975"]-valid["a_mean"]],
                fmt="none", color="black", capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels([f"G{r['group_id']:.0f}" for _, r in valid.iterrows()], rotation=45)
    ax.set_ylabel("Boundary a")
    ax.set_title("决策边界 a")

ax = axes[1, 1]
if "t_mean" in df_params.columns:
    valid = df_params.dropna(subset=["t_mean"])
    x = np.arange(len(valid))
    ax.bar(x, valid["t_mean"], color="coral", alpha=0.8)
    ax.errorbar(x, valid["t_mean"],
                yerr=[valid["t_mean"]-valid["t_q025"], valid["t_q975"]-valid["t_mean"]],
                fmt="none", color="black", capsize=3)
    ax.set_xticks(x)
    ax.set_xticklabels([f"G{r['group_id']:.0f}" for _, r in valid.iterrows()], rotation=45)
    ax.set_ylabel("t0 (s)")
    ax.set_title("非决策时间 t0")

plt.tight_layout()
plt.savefig(FIG_DIR / "ddm_params_by_group.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# -------- SPE vs 实验设计参数 --------
if "SPE_v" in df_params.columns:
    fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
    for i, (var, xlabel) in enumerate([
        ("P", "练习次数 P"),
        ("T_ms", "刺激呈现时间 T (ms)"),
        ("W_ms", "反应窗口 W (ms)"),
    ]):
        ax = axes2[i]
        valid = df_params.dropna(subset=["SPE_v", var])
        ax.scatter(valid[var], valid["SPE_v"], s=80, c="steelblue", edgecolors="black")
        for _, r in valid.iterrows():
            ax.annotate(f"G{r['group_id']:.0f}", (r[var], r["SPE_v"]),
                        textcoords="offset points", xytext=(5, 5), fontsize=8)
        ax.axhline(y=0, color="gray", linestyle="--")
        ax.set_xlabel(xlabel)
        ax.set_ylabel("SPE_v")
        ax.set_title(f"SPE_v vs {xlabel}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "spe_vs_design_params.png", dpi=200, bbox_inches="tight")
    plt.show()

print(f"\n✅ 图表已保存到: {FIG_DIR}")
print(f"✅ 参数表: {TRACE_DIR / 'all_groups_ddm_params.csv'}")

---
## 输出文件清单

| 文件 | 内容 |
|------|------|
| `2_Data/Real_Data/HDDM_Ready/hddm_data_group*.csv` | 预处理后的 HDDM 输入 |
| `2_Data/Real_Data/HDDM_Traces/*_traces.npz` | MCMC 采样迹线 |
| `2_Data/Real_Data/HDDM_Traces/*_stats.csv` | 各参数后验摘要 |
| `2_Data/Real_Data/HDDM_Traces/all_groups_ddm_params.csv` | 所有条件汇总 |
| `3_Figures/HDDM_Results/ddm_params_by_group.png` | 参数柱状图 |
| `3_Figures/HDDM_Results/spe_vs_design_params.png` | SPE vs 设计参数散点图 |

---
## ⚠️ 注意事项

1. **Group 1 & 2 遗漏率较高** (W=300/600ms + T=30ms)，DDM 拟合可能不收敛
2. 遗漏试次的 RT 设为 `T+W`(截止时间)，HDDM 应识别为截尾数据
3. 若某组 MCMC 未收敛，检查该组 `_stats.csv` 的 R-hat 值 (>1.1 需警惕)